<a href="https://colab.research.google.com/github/TiferetChayahShalom/DI_Bootcamp/blob/main/Week13/Day2/Tiferet__Exercises_XP_Day4_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: LoRA Implementation Lab
Replace each `TODO` before running the next section.

## What you'll learn

- The fundamentals of LoRA (Low-Rank Adaptation) and why it helps churn out efficient fine-tunes.
- How to implement LoRA matrices `A` and `B`, plus how to wrap existing `nn.Linear` layers.
- Differences between standard linear layers, LoRA-enhanced layers, and merged-weight alternatives.
- How to freeze base parameters so that only the LoRA adapters receive updates.

## What you will create

- A reusable `LoRALayer` module and two linear wrappers (`LinearWithLoRA`, `LinearWithLoRAMerged`).
- A 3-layer MLP that can be swapped between standard and LoRA-enhanced variants.
- A minimal MNIST training loop plus accuracy helpers to compare frozen vs. fully-trainable adapters.
- A workflow to freeze baseline weights and fine-tune only the LoRA layers.

> **Learning point**  
> Keep the student and teacher notebooks open side by side. Follow the numbered exercises, run setup only once, and watch tensor shapes as you add LoRA adapters.

# Part 0: Environment Setup

Install the CPU-friendly PyTorch stack plus torchvision for MNIST. Reuse caches across reruns to save time.

In [1]:
%pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [2]:
import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

BASE_SEED = 123
torch.manual_seed(BASE_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


# Exercise 1: Implement `LoRALayer`

Create the low-rank matrices `A` and `B`, scale them with `alpha`, and test the module on a toy tensor.

In [20]:
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.rank = rank
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        x = (x @ self.A @ self.B) * (self.alpha / self.rank)
        return x

# Hyperparameters for the sandbox test
random_seed = 123
in_dim = 10
out_dim = 5
rank = 2
alpha = 4

torch.manual_seed(random_seed)
layer = LoRALayer(in_dim, out_dim, rank, alpha)
x = torch.randn(3, in_dim)
output  = layer(x)

print("Input shape", x.shape)
print("Output shape:", output.shape)
print("Output:", output)

Input shape torch.Size([3, 10])
Output shape: torch.Size([3, 5])
Output: tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]], grad_fn=<MulBackward0>)


# Exercise 2: Wrap `nn.Linear` with LoRA

Combine a frozen linear projection plus a trainable `LoRALayer`. Confirm the adapter outputs add on top of the base logits.

In [21]:
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )
        for param in self.linear.parameters():
            param.requires_grad = False

    def forward(self, x):
        return self.linear(x) + self.lora(x)

base_linear = nn.Linear(in_dim, out_dim)
layer_lora_1 = LinearWithLoRA(base_linear, rank, alpha)
print("LinearWithLoRA output:", layer_lora_1(x))

LinearWithLoRA output: tensor([[-0.2601, -0.0784,  0.2624, -1.1216, -0.2571],
        [ 0.9431, -1.6008, -0.9093,  0.6529,  0.9265],
        [ 0.2848, -0.1656,  0.0615, -0.4130, -0.3552]], grad_fn=<AddBackward0>)


In [23]:
for name, param in layer_lora_1.named_parameters():
  print(name, param.requires_grad)

linear.weight False
linear.bias False
lora.A True
lora.B True


# Exercise 3: Swap a simple network layer with LoRA

Start from a single-layer perceptron, then replace its linear block with `LinearWithLoRA`. The outputs should match before training because the LoRA adapters start at zero.

In [6]:
class SingleLayerNet(nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()
        self.layer = nn.Linear(num_features, num_classes)

    def forward(self, x):
        return self.layer(x)

single_net = SingleLayerNet(num_features=10, num_classes=5)
sample_input = torch.randn(3,10)

with torch.no_grad():
    baseline_output = single_net(sample_input)

single_net.layer = LinearWithLoRA(single_net.layer, rank=2, alpha=4) # replace with LinearWithLoRA

with torch.no_grad():
    lora_output = single_net(sample_input)

print("Outputs match before training?", torch.allclose(baseline_output, lora_output))

Outputs match before training? True


# Exercise 4: Merged-weight LoRA layer

Fuse the LoRA matrices with the frozen weights to create a drop-in linear layer that behaves exactly like `LinearWithLoRA`.

In [9]:
import torch.nn.functional as F

class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        lora = self.lora.A @ self.lora.B
        combined_weight = self.linear.weight + (lora * (self.lora.alpha / self.lora.A.shape[1])).T  # merge original weights + scaled LoRA correction
        return F.linear(x, combined_weight, self.linear.bias)

layer_lora_2 = LinearWithLoRAMerged(nn.Linear(in_dim, out_dim), rank, alpha)
print("Merged LoRA output:", layer_lora_2(x))

Merged LoRA output: tensor([[ 0.2312, -0.0273,  0.2762, -0.3550, -0.3793],
        [-0.6399,  0.9192, -0.9796, -0.1069,  0.1556],
        [ 0.4765,  0.0832,  0.0698, -0.3171, -0.6309]],
       grad_fn=<AddmmBackward0>)


# Exercise 5: Build an MLP and prepare MNIST

Stack three linear layers with ReLU activations, then set up the MNIST loaders plus optimizer/state for pretraining.

In [10]:
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_features, num_hidden_1),
            nn.ReLU(),
            nn.Linear(num_hidden_1, num_hidden_2),
            nn.ReLU(),
            nn.Linear(num_hidden_2, num_classes)
        )

    def forward(self, x):
        x = self.layers(x)
        return x

In [12]:
# Architecture
num_features = 784
num_hidden_1 = 128
num_hidden_2 = 64
num_classes = 10

# Settings
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
learning_rate = 0.001
num_epochs = 10

model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes,
)

model.to(DEVICE)
optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)
print(DEVICE)
print(model)
print(optimizer_pretrained)

cpu
MultilayerPerceptron(
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


## Loading dataset

In [13]:
BATCH_SIZE = 64

train_dataset = datasets.MNIST(root='data', train=True, transform=transforms.ToTensor(), download=True)

test_dataset = datasets.MNIST(root='data', train=False, transform=transforms.ToTensor(), download=True)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

for images, labels in train_loader:
    print('Image batch dimensions:', images.shape)
    print('Image label dimensions:', labels.shape)
    break

100%|██████████| 9.91M/9.91M [00:00<00:00, 65.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.72MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.4MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.4MB/s]


Image batch dimensions: torch.Size([64, 1, 28, 28])
Image label dimensions: torch.Size([64])


## Define evaluation

In [30]:
def compute_accuracy(model, data_loader, device):
    was_training = model.training
    model.eval()
    correct_pred, num_examples = 0, 0
    with torch.no_grad():
        for features, targets in data_loader:
            features = features.view(features.size(0), -1).to(device)
            targets = targets.to(device)
            logits = model(features)
            _, predicted_labels = torch.max(logits, 1)
            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum().item()
    model.train(was_training)
    return (correct_pred / num_examples) * 100

## Training

In [31]:
def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):
            features = features.view(features.size(0), -1).to(device)
            targets = targets.to(device)

            logits = model(features)
            loss = F.cross_entropy(logits, targets)
            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            if not batch_idx % 400:
                print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' % (epoch+1, num_epochs, batch_idx, len(train_loader), loss))

        with torch.set_grad_enabled(False):
            print('Epoch: %03d/%03d training accuracy: %.2f%%' % (epoch+1, num_epochs, compute_accuracy(model, train_loader, device)))

        print('Time elapsed: %.2f min' % ((time.time() - start_time)/60))
    print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))

In [32]:
train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

Epoch: 001/010|Batch 000/938| Loss: 0.0302
Epoch: 001/010|Batch 400/938| Loss: 0.0056
Epoch: 001/010|Batch 800/938| Loss: 0.1074
Epoch: 001/010 training accuracy: 99.43%
Time elapsed: 0.29 min
Epoch: 002/010|Batch 000/938| Loss: 0.0062
Epoch: 002/010|Batch 400/938| Loss: 0.0011
Epoch: 002/010|Batch 800/938| Loss: 0.0035
Epoch: 002/010 training accuracy: 99.68%
Time elapsed: 0.59 min
Epoch: 003/010|Batch 000/938| Loss: 0.0152
Epoch: 003/010|Batch 400/938| Loss: 0.1062
Epoch: 003/010|Batch 800/938| Loss: 0.0622
Epoch: 003/010 training accuracy: 99.70%
Time elapsed: 0.87 min
Epoch: 004/010|Batch 000/938| Loss: 0.0116
Epoch: 004/010|Batch 400/938| Loss: 0.0006
Epoch: 004/010|Batch 800/938| Loss: 0.0010
Epoch: 004/010 training accuracy: 99.62%
Time elapsed: 1.15 min
Epoch: 005/010|Batch 000/938| Loss: 0.0013
Epoch: 005/010|Batch 400/938| Loss: 0.0000
Epoch: 005/010|Batch 800/938| Loss: 0.0269
Epoch: 005/010 training accuracy: 99.63%
Time elapsed: 1.45 min
Epoch: 006/010|Batch 000/938| Loss:

# Replacing Linear with LoRA Layers

In [33]:
model_lora = copy.deepcopy(model)

model_lora.layers[0] = LinearWithLoRAMerged(model_lora.layers[0], rank=4, alpha=8)
model_lora.layers[2] = LinearWithLoRAMerged(model_lora.layers[2], rank=4, alpha=8)
model_lora.layers[4] = LinearWithLoRAMerged(model_lora.layers[4], rank=4, alpha=8)
model_lora.to(DEVICE)
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
print(model_lora)

print(f'Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

MultilayerPerceptron(
  (layers): Sequential(
    (0): LinearWithLoRAMerged(
      (linear): Linear(in_features=784, out_features=128, bias=True)
      (lora): LoRALayer()
    )
    (1): ReLU()
    (2): LinearWithLoRAMerged(
      (linear): Linear(in_features=128, out_features=64, bias=True)
      (lora): LoRALayer()
    )
    (3): ReLU()
    (4): LinearWithLoRAMerged(
      (linear): Linear(in_features=64, out_features=10, bias=True)
      (lora): LoRALayer()
    )
  )
)
Test accuracy orig model:97.57%
Test accuracy LoRA model:97.57%


## Freezing the Original Linear Layers

In [34]:
def freeze_linear_layers(model):
    for child in model.children():
        if isinstance(child, nn.Linear):
            for param in child.parameters():
                param.requires_grad = False
        else:
            freeze_linear_layers(child)

freeze_linear_layers(model_lora)
for name, param in model_lora.named_parameters():
    print(f'{name}:{param.requires_grad}')

layers.0.linear.weight:False
layers.0.linear.bias:False
layers.0.lora.A:True
layers.0.lora.B:True
layers.2.linear.weight:False
layers.2.linear.bias:False
layers.2.lora.A:True
layers.2.lora.B:True
layers.4.linear.weight:False
layers.4.linear.bias:False
layers.4.lora.A:True
layers.4.lora.B:True


In [ ]:
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)
print(f'Test accuracy LoRA finetune: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

print(f'Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

Epoch: 001/010|Batch 000/938| Loss: 0.0080
Epoch: 001/010|Batch 400/938| Loss: 0.0008
Epoch: 001/010|Batch 800/938| Loss: 0.0024
Epoch: 001/010 training accuracy: 99.95%
Time elapsed: 0.30 min
Epoch: 002/010|Batch 000/938| Loss: 0.0009
Epoch: 002/010|Batch 400/938| Loss: 0.0000
Epoch: 002/010|Batch 800/938| Loss: 0.0001
Epoch: 002/010 training accuracy: 99.97%
Time elapsed: 0.59 min
Epoch: 003/010|Batch 000/938| Loss: 0.0031
Epoch: 003/010|Batch 400/938| Loss: 0.0019
Epoch: 003/010|Batch 800/938| Loss: 0.0000
Epoch: 003/010 training accuracy: 99.97%
Time elapsed: 0.88 min
Epoch: 004/010|Batch 000/938| Loss: 0.0007
Epoch: 004/010|Batch 400/938| Loss: 0.0000
Epoch: 004/010|Batch 800/938| Loss: 0.0004
Epoch: 004/010 training accuracy: 99.97%
Time elapsed: 1.17 min
Epoch: 005/010|Batch 000/938| Loss: 0.0113
Epoch: 005/010|Batch 400/938| Loss: 0.0014
Epoch: 005/010|Batch 800/938| Loss: 0.0002
Epoch: 005/010 training accuracy: 99.96%
Time elapsed: 1.47 min
Epoch: 006/010|Batch 000/938| Loss: